In [ ]:
# import toml
# from pathlib import Path
# config_path = Path("../../../../configuration/asim_configuration")
# input_config = toml.load(config_path/ "input_configuration.toml")
# summary_config = toml.load(config_path/ "summary_configuration.toml")

For all workers that work out of the home, the telecommute models predicts the level of telecommuting. The model alternatives are the frequency of telecommuting in days per week (0 days, 1 day, 2 to 3 days, 4+ days).

Telecommuting is defined as workers who work from home instead of going to work. It only applies to workers with a regular workplace outside of home.

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

In [ ]:
# read data
person = util.get_person_data(summary_config, uncloned=True)
hh_data = util.get_hh_data(summary_config, uncloned=True)

per_data = person.\
    merge(util.get_landuse_data(summary_config)[['zone_id', 'TAZ', 'county_label']], how="left", left_on='home_zone_id', right_on='zone_id').\
    merge(hh_data[['source','household_id','income_group']], how="left", on=['source','household_id'])

# get workers that are not working from home
per_data = per_data.loc[(per_data['pemploy']<3) & (~per_data['work_from_home'])]


In [5]:
util.plot_share_barchart(per_data, 'person_weight', 'telecommute_frequency_label', title="telecommute frequency<br>for workers with usual work location outside of home")

## telecommute frequency by segment

In [7]:
# auto ownership in Income groups
def plot_segments(df:pd.DataFrame, var:str, title_cat:str,sub_name:str):
    # print(f"n=\n"
    #       f"{df.loc[df['source']=='model results',var].value_counts()[df[var].sort_values().unique()]}")
    df_plot = df.groupby(['source',var,'telecommute_frequency_label'])['person_weight'].sum().reset_index()
    df_plot['percentage'] = df_plot.groupby(['source',var], group_keys=False)['person_weight'].\
        apply(lambda x: 100 * x / float(x.sum()))

    fig = px.bar(df_plot, x="telecommute_frequency_label", y="percentage", color="source",
                 facet_col=var, barmode="group",
                 title="telecommute frequency by "+ title_cat)
    fig.for_each_annotation(lambda a: a.update(text = sub_name + "=<br>" + a.text.split("=")[-1]))
    fig.update_xaxes(title_text="")
    fig.update_layout(height=400, width=800, font=dict(size=11))
    fig.show()

In [8]:
util.plot_share_facetbar(per_data, 'person_weight', share_col="telecommute_frequency_label", title="telecommute frequency by household income group", 
                            facet_col="income_group", facet_col_wrap=4, height=400, orientation='v')

In [ ]:
util.plot_share_facetbar(per_data, 'person_weight', share_col="telecommute_frequency_label", title="telecommute frequency by county", 
                            facet_col="county_label", facet_col_wrap=4, height=400, orientation='v')